only accepting full double barrel scientific names and variants no only genus or only species names
# This is deprecated, it was never tested but was designed to normalize plants using powo then add to db the new paradigm will be to add powo to db and the incoming plant compound relationships will be added
## **Bulk plant normalizer**
#### **This plant normalizer should do the following:**
- take in and format the wcvp
- take in a file retured from the checklist bank that has been deduplicated on the origional name
- keep only the GBIF name, ID, accepted name, accepted ID
- remove non-plants
- remove any genus or species only names (need both genus and species minimum)
- adding a cleaned name and matching the datasets based on that 
- fuzzy matching on the cleaned names using the fuzz.ratio scorer
- linking back to accepted ID
- collapsing into unified main ID with all else going to synonyms

In [ ]:
wcvp_file = ''
new_checklisted_file = ''
staging_table= ''
wand = ''
error_output_path = ''
output_path = ''
report_path = ''
wcvp_locations = ''
#////////////////////////////////////////////////////////imports
import pandas as pd
from rapidfuzz import process
import re
#////////////////////////////////////////////////////////functions
def get_name_clean(name,id):
    global errors
    if pd.isna(name):
        err = [id, 'Name cleaning error', 'There was no name for this entry']
        errors.loc[len(errors)] = err
        return pd.NA
    if 'sp' in name or 'spp' in name:
        err = [id, 'Name error', f'{name} contains "sp" or "spp" and was therefore excluded']
        errors.loc[len(errors)] = err
        return pd.NA
    name = name.strip().lower().replace('-','').split()
    name = ' '.join(name)    
    return name

def is_complete_name(name,id):
    global errors
    og_name = name
    if pd.isna(name) or name == '' or not name.strip():
        err = [id,'Incomplete name','There was no name for this entry']
        errors.loc[len(errors)] = err
        return False
    else:
        name = name.strip()
        if len(name.split()) < 2:
            return True
        else:
            err = [id,'Incomplete name',f'{og_name} is not a valid name (likely was only genus or species)']
            errors.loc[len(errors)] = err
            return False

def fuzzy_match(og_name, og_id, name_id_dict):
    cutoff = 95
    result = process.extractOne(og_name,name_id_dict.keys())
    if result is None:
        err = [og_id, 'Fuzzy match error', f'Failed direct match and fuzzy match with {og_name} matching to none of the wcvp']
        errors.loc[len(errors)] = err
        return pd.NA
    match, chance, _ = result
    if chance < cutoff:
        err = [og_id, 'Fuzzy match error', f'Failed direct match and fuzzy match with {og_name} matching to {match} at {chance}% chance']
        errors.loc[len(errors)] = err
        return pd.NA
    else:
        return name_id_dict.get(match)
    
def format_wcvp(df):
    df = df.copy()
    df = df.rename(columns={'scientfiicname':'scientific_name','scientfiicnameauthorship':'scientific_name_authorship'}, ignore_errors = True)
    df.drop(columns=['genus','specificepithet','infraspecificepithet','taxonrank','taxonomicstatus','nomenclaturalstatus','taxonremarks'],inplace=True)
    df['scientific_name'] = df.apply(lambda row: clean_name(row['scientific_name']),axis = 1)
    return df

def format_check_new_plants(df):
    df = df.copy()
    columns  = df.columns
    if 'origional_scientific_name' not in columns or 'original_id' not in columns or 'kigdom' not in columns or 'ID' not in 'columns' or 'scientificName' not in columns or 'acceptedScientificName' not in columns or 'acceptedID' not in columns:
        print('Terminated the input file was missing columns')
        raise Exception('The input file does not have all the requisite columns')
    df = df['orginal_scientific_name','original_id', 'kingdom', 'ID', 'scientificName', 'acceptedScientificName', 'acceptedID'] 
    return df

def combine_dedup(items):
    items = [item.strip() for item in items if item.strip()]
    out = []
    for item in items:
        if item not in out:
            out.append(item)
    return '|'.join(out) 

def clean_name(name):
    if pd.isna(name):
        return pd.NA
    name = name.strip().lower().replace('-','').split()
    name = ' '.join(name)    
    return name

def get_wcvp_syns_powo_ids(taxonid):
    a = wcvp.loc[wcvp['acceptedID'] == taxonid].copy()
    a_syns = a['scientific_name'].to_list()
    a_powo = a['references'].to_list()

    og = wcvp.loc[wcvp['originalNameUsageID'] == taxonid].copy()
    og_syns = og['scientific_name'].to_list()
    og_powo = og['references'].to_list()
    
    syns = list(a_syns + og_syns)
    syns = [syn.strip() for syn in syns if syn.strip()]
    out = []
    for syn in syns:
        if syn not in out:
            out.append(syn)

    powo = list(a_powo + og_powo)
    powo = (p.split(':')[0] for p in powo)
    powo = [p.strip() for p in powo if p.strip()]
    out_powo = []
    for p in powo:
        if p not in out_powo:
            out_powo.append(p)
    return '|'.join(out), '|'.join(out_powo)

def get_wcvp_accepted_name(taxonid):
    accepted_name = wcvp.loc[wcvp['taxon_id'] == taxonid, 'scientific_name'].values[0]
    return accepted_name

def remove_name_from_syns(syns, name):
    if not syns:
        return pd.NA
    syns = [syn for syn in syns if syn != name]
    return '|'.join(syns)

def add_og_gbif_name_to_syns(syns, og_name):
    syns = syns.split('|')
    syns.append(og_name)
    syns = [syn.strip() for syn in syns if syn.strip()]
    out = []
    for syn in syns:
        if syn not in out:
            out.append(syn)
    return '|'.join(out)

def get_wcvp_accepted_id(taxonid):
    accepted_id = wcvp.loc[wcvp['taxonid'] == taxonid, 'acceptedID'].values[0]
    if pd.isna(accepted_id):
        return taxonid
    return accepted_id

def get_wcvp_accepted_authorship(taxonid):
    accepted_authorship = wcvp.loc[wcvp['taxonid'] == taxonid, 'scientific_name_authorship'].values[0]
    return accepted_authorship

def get_wcvp_name_published_in(taxonid):
    name_published_in = wcvp.loc[wcvp['taxonid'] == taxonid, 'namepublishedin'].values[0]
    return name_published_in

def genus_species_splitter(name):
    if pd.isna(name): 
        return pd.NA, pd.NA
    genus = name.split()[0]
    species = ' '.join(name.split()[1:])
    return genus, species

def get_wand_family_id(family,family_dict):
    if family in family_dict:
        return family_dict[family]
    else:
        raise Exception(f'Family {family} not found in wand family dictionary')

def get_use_taxon_id(taxonid):
    use_taxon_id = wcvp.loc[wcvp['taxonid'] == taxonid, 'useTaxonID'].values[0]
    if pd.isna(use_taxon_id):
        return taxonid
    return use_taxon_id

def format_new_for_db(df):
    df = df.copy()
    if wand_plants.empty:
        start_id = 1
    else:
        start_id = pd.to_numeric(wand_plants['plant_id']).max()+1

    df['use_id'] = df.apply(lambda row: get_use_taxon_id(row['taxonid']),axis=1)
    df['powo_id'] = df.apply(lambda row: get_wcvp_accepted_id(row['taxonid']),axis=1)
    df['plant_id'] = range(start_id, start_id+len(df))
    df['plant_name'] = df.apply(lambda row: get_wcvp_accepted_name(row['use_id']),axis=1)
    df['synonyms'], df['syn_powo_id'] = zip(*df.apply(lambda row: get_wcvp_syns_powo_ids(row['use_id']),axis=1))
    df['synonyms'] = df.apply(lambda row: remove_name_from_syns(row['synonyms'], row['plant_name']),axis=1)
    df['plant_authorship'] = df.apply(lambda row: get_wcvp_accepted_authorship(row['use_id']),axis=1)
    df['plant_published_in'] = df.apply(lambda row: get_wcvp_name_published_in(row['use_id']),axis=1)
    df = df.apply(lambda row: add_og_gbif_name_to_syns(row['synonyms'], row['original_scientificName']),axis=1)
    df = df.apply(lambda row: add_og_gbif_name_to_syns(row['synonyms'], row['scientificName']),axis=1)
    df['genus'], df['species'] = zip(*df.apply(lambda row: genus_species_splitter(row['plant_name']),axis=1))
    df['family_id'] = df.apply(lambda row: get_wand_family_id(row['family'], family_dict),axis=1)
    df.rename(columns={'acceptedID':'gbif_accepted_id'})
    df = df[['plant_id','plant_name','genus','species','synonyms','powo_id','gbif_accepted_id','plant_authorship','plant_published_in','family_id']]
    return df   

#////////////////////////////////////////////////////////data
                                #////////////////////////load data
print('Loading data...', flush = True)
wcvp = pd.read_csv(wcvp_file, sep='|')
new = pd.read_csv(new_checklisted_file)
errors = pd.DataFrame(columns=['id','error_stage','exact_error'])
wand_plants = pd.read_csv(wand)
                                #////////////////////////format wcvp and new
print('Formatting and checking wcvp and new plants...', flush = True)
wcvp = format_wcvp(wcvp)
new_size = len(new)
new = format_check_new_plants(new)

#////////////////////////////////////////////////////////normalize new plants
                                #////////////////////////removing non-plants
print('Removing non-plants from gbif match...', flush = True)
is_plant = new['kingdom'].eq('Plantae').fillna(False)
new_plants = new.loc[is_plant].copy()
non_plants = new.loc[~is_plant].copy()
no_kingdoms = non_plants.loc[non_plants['kingdom'].isna()].copy()
no_kingdoms_err = pd.DataFrame({'id': no_kingdoms['origional_id'],'error_stage': 'No gbif kingdom', 'exact_error': (no_kingdoms['original_scientificName'] + ' entry excluded there was no kingdom')})
errors.append(no_kingdoms_err)
non_plants = non_plants.loc[non_plants['kingdom'].notna()].copy()
non_plant_errors  = pd.DataFrame({'id': non_plants['origional_id'],'error_stage': 'GBIF kingdom was not plantae', 'exact_error': (non_plants['original_scientificName'] + ' not recognised by gbif as a plant')})
errors = errors.append(non_plant_errors)
                                #////////////////////////removing incomplete names
print('Removing incomplete names...', flush = True)
new_plants['name_complete'] = new_plants.apply(lambda row: is_complete_name(row['original_scientificName'],row['origional_id']),axis = 1)
new_plants = new_plants.loc[new_plants['name_complete']].copy()
                                #////////////////////////adding clean name for direct matching
print('Name cleaning...')
new_plants['clean_name'] = new_plants.apply(lambda row: get_name_clean(row['original_scientificName'],row['origional_id']),axis = 1)
new_plants.dropna(subset=['clean_name'],inplace=True)

                                #////////////////////////checking for duplicates on clean name and removing if necessary
if new_plants.duplicated(subset='clean_name').any():
    print('Once the names were cleaned for direct original name wcvp matching some of the cleaned names were duplicated the script keeps the first and removes the rest of the duplicates. Please ensure the original file and the checklist return file are deduplicated on the plant name. I this was done then the actual cleaning process leads to duplicates of names by removing "-" and whitespace as well as comparing only lowercase. This means redundancy was likely removed. The removed entries are in the error file. \n', flush = True)
    pre_dedup_size = len(new_plants)
    new_plants = new_plants.groupby('clean_name',as_index=False,sort=False).agg({column: combine_dedup for column in new_plants.columns})
    post_dedup_size = len(new_plants)
                                #////////////////////////seperating new plants from plants in the db already

                                #////////////////////////matching cleaned names
print('Matching cleaned names to wcvp...', flush = True)
if new_plants['clean_name'].str.contains("|", regex=False, na=False).any():
    print('| combined clean name', flush = True)
    raise Exception('| combined clean name')
pre_match_size = len(new_plants)
new_plants = new_plants.merge(wcvp, how='left',left_on='clean_name',right_on = 'scientific_name', validate='one_to_one')
                                #////////////////////////fuzzy matching what is left
print('Fuzzy matching unmatched cleaned names to wcvp...', flush = True)
unmatched = new_plants.loc[new_plants['taxonid'].isna()].copy()
wcvp_name_check_dic = dict(zip(wcvp['scientific_name'],wcvp['taxonid']))
unmatched['taxonid_fuzzy'] = unmatched.apply(lambda row: fuzzy_match(row.clean_name, row.original_id, wcvp_name_check_dic),axis=1)
                                #////////////////////////merging fuzzy matched to the wcvp
unmatched = unmatched[['orginal_scientific_name','original_id', 'kingdom', 'ID', 'scientificName', 'acceptedScientificName', 'acceptedID','taxonid_fuzzy']]
unmatched = unmatched.merge(wcvp, how='left',left_on='taxonid_fuzzy',right_on = 'taxonid', validate='one_to_one')
                                #////////////////////////combining the direct matched and fuzzy matched into one df and adding a match type column
unmatched.drop(columns=['taxonid_fuzzy'],inplace=True)
unmatched['match_type'] = 'fuzzy match'
new_plants['match_type'] = 'direct_match'
#there is no need for an error handler here to add the dropped as they were added already during fuzzy match failure
new_plants.dropna(subset='taxonid',inplace=True)
new_plants = pd.concat([new_plants,unmatched],ignore_index=True)
num_direct_matches = new_plants['match_type'].eq('direct_match').sum()
num_fuzzy_matches = new_plants['match_type'].eq('fuzzy match').sum()
num_unmatched = pre_match_size - num_direct_matches - num_fuzzy_matches
                                #////////////////////////getting accepted names and adding synonyms formatting for wand
print('Formatting for wand...', flush = True)
new_plants = format_for_db(new_plants)
                                #////////////////////////saving new
print('Saving new plants to csv...', flush = True)
new_plants.to_csv(output_path, index=False)
#////////////////////////////////////////////////////////plants already in db

#////////////////////////////////////////////////////////report
new_size = f'There were {new_size:,} new plants in the input file'.replace(',',' ')
stats = new_size + '\n'
if pre_dedup_size:
    dedup_change = pre_dedup_size - post_dedup_size
    pre_dedup_size = f'{pre_dedup_size:,}'.replace(',', ' ')
    post_dedup_size = f'{post_dedup_size:,}'.replace(',', ' ')
    dedup_change = f'{dedup_change:,}'.replace(',',' ')
    dedup_msg = f"""New compounds before deduplication: {pre_dedup_size} \n
                    New compounds after deduplication: {post_dedup_size} \n
                    New compounds removed during deduplication: {dedup_change} \n"""
    stats += dedup_msg
num_direct_mathches = f'{num_direct_matches:,}'.replace(',',' ')
num_fuzzy_matches = f'{num_fuzzy_matches:,}'.replace(',',' ')
num_unmatched = f'{num_unmatched:,}'.replace(',',' ')
stats = f"""The number matched directly: {num_direct_matches:,} \n
            The number fuzzy matched: {num_fuzzy_matches:,}\n 
            The number unmatched: {num_unmatched:,}\n"""

item = error_messages + '\n' + stats
with open(report_path, 'w') as f:
    f.write(item)
#////////////////////////////////////////////////////////output

# Family dictionary maker

In [10]:
import pandas as pd

wcvp = pd.read_csv('/home/school/masters/Scripts/wcvp_dwca/wcvp_taxon.csv',sep='|',usecols=['taxonid','taxonomicstatus' ,'acceptednameusageid','family','scientfiicname','taxonrank','taxonrank','originalnameusageid','references'])

families = wcvp.loc[wcvp['taxonomicstatus'].eq('Accepted')].copy()
families = families.drop_duplicates(subset='family')
print(len(families))

458


# Location table

- just making the actual table for the db I can come back and add the links to the plants later using this as a basis

In [24]:
import pandas as pd

def remover(code):
    code = code[5:]
    return code

df = pd.read_csv('/home/school/masters/Scripts/wcvp_dwca/wcvp_distribution.csv', sep = '|')

df.drop_duplicates(subset=['locality','locationid'],inplace=True)

locations = df.copy()

locations = locations[['locality','locationid']]

locations = locations.rename(columns={'locationid': 'code'})

locations['location_id'] = range(1,len(locations)+1)

locations['code']  = locations.apply(lambda row: remover(row['code']),axis = 1)

locations = locations[['location_id','locality','code']]

locations.to_csv('/home/school/masters/Scripts/wcvp_dwca/distributions_for_db.csv',index = False)



# POWO adder updater

the idea is that this will take in 3 files the powo, distributions and a file with a kew backbone and then regenerate the whole plant thing off of that

POWO gbif matched -> has gbif accepted ID

My implementation here is probably a little strange because I am using code from above but it should save me time so yeah I am using the powo wcvp to format the powo wcvp

In [ ]:
#///////////////////////////////////input files
powo_file = ''
taxonomic_backbone_file = ''
distrbutions_file = '' # expects families .tree file from tree of life kew 
#/////////////////////////////////// output files
powo_for_db = ''
distribution_for_db = ''
plant_distribution_junction = ''
#/////////////////////////////////// imports
from Bio import Phylo
import pandas as pd
import pykew.ipni as ipni
import requests
from bs4 import BeautifulSoup
#///////////////////////////////////POWO
#Functions
def format_wcvp(df):
    df = df.copy()
    df = df.rename(columns={'scientfiicname':'scientific_name','scientfiicnameauthorship':'scientific_name_authorship'}, ignore_errors = True)
    df.drop(columns=['genus','specificepithet','infraspecificepithet','taxonrank','taxonomicstatus','nomenclaturalstatus','taxonremarks'],inplace=True)
    df['scientific_name'] = df.apply(lambda row: clean_name(row['scientific_name']),axis = 1)
    return df

def format_wcvp(df):
    df = df.copy()
    df = df.rename(columns={'scientfiicname':'scientific_name','scientfiicnameauthorship':'scientific_name_authorship'}, ignore_errors = True)
    df.drop(columns=['genus','specificepithet','infraspecificepithet','taxonrank','taxonomicstatus','nomenclaturalstatus','taxonremarks'],inplace=True)
    df['scientific_name'] = df.apply(lambda row: clean_name(row['scientific_name']),axis = 1)
    return df

def combine_dedup(items):
    items = [item.strip() for item in items if item.strip()]
    out = []
    for item in items:
        if item not in out:
            out.append(item)
    return '|'.join(out) 

def get_wcvp_syns_powo_ids(taxonid):
    a = wcvp.loc[wcvp['acceptedID'] == taxonid].copy()
    a_syns = a['scientific_name'].to_list()
    a_powo = a['references'].to_list()

    og = wcvp.loc[wcvp['originalNameUsageID'] == taxonid].copy()
    og_syns = og['scientific_name'].to_list()
    og_powo = og['references'].to_list()
    
    syns = list(a_syns + og_syns)
    syns = [syn.strip() for syn in syns if syn.strip()]
    out = []
    for syn in syns:
        if syn not in out:
            out.append(syn)

    powo = list(a_powo + og_powo)
    powo = (p.split(':')[0] for p in powo)
    powo = [p.strip() for p in powo if p.strip()]
    out_powo = []
    for p in powo:
        if p not in out_powo:
            out_powo.append(p)
    return '|'.join(out), '|'.join(out_powo)

def get_wcvp_accepted_name(taxonid):
    accepted_name = wcvp.loc[wcvp['taxon_id'] == taxonid, 'scientific_name'].values[0]
    return accepted_name

def remove_name_from_syns(syns, name):
    if not syns:
        return pd.NA
    syns = [syn for syn in syns if syn != name]
    return '|'.join(syns)

def add_og_gbif_name_to_syns(syns, og_name):
    syns = syns.split('|')
    syns.append(og_name)
    syns = [syn.strip() for syn in syns if syn.strip()]
    out = []
    for syn in syns:
        if syn not in out:
            out.append(syn)
    return '|'.join(out)

def get_wcvp_accepted_id(taxonid):
    accepted_id = wcvp.loc[wcvp['taxonid'] == taxonid, 'acceptedID'].values[0]
    if pd.isna(accepted_id):
        return taxonid
    return accepted_id

def get_wcvp_accepted_authorship(taxonid):
    accepted_authorship = wcvp.loc[wcvp['taxonid'] == taxonid, 'scientific_name_authorship'].values[0]
    return accepted_authorship

def get_wcvp_name_published_in(taxonid):
    name_published_in = wcvp.loc[wcvp['taxonid'] == taxonid, 'namepublishedin'].values[0]
    return name_published_in

def genus_species_splitter(name):
    if pd.isna(name): 
        return pd.NA, pd.NA
    genus = name.split()[0]
    species = ' '.join(name.split()[1:])
    return genus, species

def get_wand_family_id(family,family_dict):
    if family in family_dict:
        return family_dict[family]
    else:
        raise Exception(f'Family {family} not found in wand family dictionary')

def get_use_taxon_id(taxonid):
    use_taxon_id = wcvp.loc[wcvp['taxonid'] == taxonid, 'useTaxonID'].values[0]
    if pd.isna(use_taxon_id):
        return taxonid
    return use_taxon_id

def is_accepted(taxonid):
    accepted = wcvp.loc[wcvp['taxonid'] == taxonid, 'acceptedUsageID'].values[0]
    if pd.notna(accepted):
        return False
    else:
        return True

def wand_format(df):
    df = df.copy()
    df['use_id'] = df.apply(lambda row: get_use_taxon_id(row['taxonid']),axis=1)
    df['powo_id'] = df.apply(lambda row: get_wcvp_accepted_id(row['taxonid']),axis=1)
    df['plant_id'] = range(1,len(df)+1)
    df['plant_name'] = df.apply(lambda row: get_wcvp_accepted_name(row['use_id']),axis=1)
    df['synonyms'], df['syn_powo_id'] = zip(*df.apply(lambda row: get_wcvp_syns_powo_ids(row['use_id']),axis=1))
    df['synonyms'] = df.apply(lambda row: remove_name_from_syns(row['synonyms'], row['plant_name']),axis=1)
    df['plant_authorship'] = df.apply(lambda row: get_wcvp_accepted_authorship(row['use_id']),axis=1)
    df['plant_published_in'] = df.apply(lambda row: get_wcvp_name_published_in(row['use_id']),axis=1)
    df = df.apply(lambda row: add_og_gbif_name_to_syns(row['synonyms'], row['original_scientificName']),axis=1)
    df = df.apply(lambda row: add_og_gbif_name_to_syns(row['synonyms'], row['scientificName']),axis=1)
    df['genus'], df['species'] = zip(*df.apply(lambda row: genus_species_splitter(row['plant_name']),axis=1))
    df.rename(columns={'acceptedID':'gbif_accepted_id'})
    df = df[['plant_id','plant_name','genus','species','synonyms','powo_id','plant_authorship','plant_published_in','family','taxonid']]
    return df 

#load powo
powo = pd.read_csv(powo_file, sep = '|')
wcvp = pd.read_csv(powo_file, sep = '|') # in order to reuse above code i am checking the powo agianst itself
#format powo
powo = format_wcvp(powo)
#get accepted only plants
powo['accepted'] = powo.apply(lambda row: is_accepted(row['taxon_id']))
powo = powo.loc[powo['is_accepted']].copy()
#wand_format
powo = wand_format(powo)
#//////////////////////////////// Distributions
#Functions
def remover(code):
    code = code[5:]
    return code
# load Distributions
df = pd.read_cs('/home/school/masters/Scripts/wcvp_dwca/vwcvp_distribution.csv', sep = '|')
#make table
df.drop_duplicates(subset=['locality','locationid'],inplace=True)
locations = df.copy()
locations = locations[['locality','locationid','core_id']]
locations = locations.rename(columns={'locationid': 'code'})
locations['location_id'] = range(1,len(locations)+1)
locations['code']  = locations.apply(lambda row: remover(row['code']),axis = 1)
locations = locations[['location_id','locality','code','coreid']]

#////////////////////////////////// Taxonomic backbone
#Functions
def get_orders_fams(tree):
    out = set()
    for clade in tree.find_clades:
        name = clade.name
        if name:
            out.add(name.split('_')[0])
    return out
#load newick
tree = Phylo.read(taxonomic_backbone_file,'newick')
#load families and orders
orders_fams = get_orders_fams(tree)
#make into table format with what is in the phylogeny
orders = pd.DataFrame(columns=['order_id','order_name'])
orders['name'] = [order.split('_')[0] for order in orders_fams]
orders.drop_duplicates(subset='order_name',inplace=True)
orders['order_id'] = range(1,len(orders)+1)
families = pd.DataFrame(columns=['family_id','family_name','order_name','order_id'])
families['family_name'] = [order.split('_')[1] for order in orders_fams]
families['order_name'] = [order.split('_')[0] for order in orders_fams]
families['order_id'] = families['order_name'].map(orders.set_index('order_name')['order_id'])
families.drop(columns=['order_name'])
orders = orders.rename(columns={'order_name':'name'})
families = families.rename(columns={'family_name': 'name'})
#////////////////////////////////// add family id to powo
powo = powo.merge(families,how='left',left_on='family',right_on='name', how='1:m')
powo.drop(columns=['name','order_id'])
#////////////////////////////////// scrape missing families
# functions
def add_missing_fams_to_families(fams):
    global families
    for fam in fams:
        id = len(families)+1
        fa = [id,fam]
        families.loc[len(families)] = fa

def get_missing_orders(fams):
    ipnis = []
    for fam in fams:
        ipnis.append(ipni.search(fam))

    orders = get_order_from_web(ipnis)
    return orders

def get_order_from_web(ipnis):
    orders = []
    for ipni in ipnis:
        url = f'https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:{ipni}'
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        div_element = soup.find('div',class_='cl-value-text')
        name = div_element.get_text()
        orders.append(name)

    return orders



#get missing families
missing_fams = powo.loc[powo['family_id'].isna(),'family'].to_list()
# add those families
add_missing_fams_to_families(missing_fams)
# get_orders
new_orders = get_missing_orders(missing_fams)

 
powo.drop(columns=['family'], inplace = True)
#////////////////////////////////// Location-plant link
#Functions 

# load
powo_for_combo = powo.copy()
distri_for_combo = pd.read_csv('/home/school/masters/Scripts/wcvp_dwca/vwcvp_distribution.csv', sep = '|')
wand_plant_distri = powo_for_combo.merge(distri_for_combo, how='left', left_on='taxon_id', right_on='core_id', validate='m:1')
wand_plant_distri = wand_plant_distri[['plant_id','location_id','establishmentmeans']]
wand_plant_distri = wand_plant_distri.rename(columns={'establishmentmeans': 'is_introduced'})
wand_plant_distri.loc[wand_plant_distri['is_introduced']=='introduced'] == 'True'
wand_plant_distri = wand_plant_distri['is_introduced'].fillna('True')
#////////////////////////////////// Final touches

#////////////////////////////////// Report

#////////////////////////////////// Save
powo.to_csv(powo_for_db,index=False)
locations.to_csv(distribution_for_db,index=False)
wand_plant_distri.to_csv(plant_distribution_junction, index=False)

testing the web scraping to get ipni code for family then scrape order from kew but unfortuantely that wonk work as pykew gives 403 at the ipni adress and the family id is not accessible via wcvp to get to the right page to scrape

In [ ]:
# testing web scraping -> did not work due to 403 error (not allowed)

from Bio import Phylo
import pandas as pd
import pykew.ipni as ipni
import requests
from bs4 import BeautifulSoup

import pykew.ipni as ipni

result = ipni.search('Poa annua')

import requests
from pykew.ipni_terms import Name
query = {Name.family: 'Asteraceae'}

# Perform the search
results = ipni.search(query)

url = "https://ipni.org"
params = {'q': 'Aspleniaceae'}

response = requests.get(url, params=params)

# 1. Check if the server returned a successful status code
if response.status_code == 200:
    try:
        data = response.json()
        print(data)
    except ValueError:
        print("Caught JSONDecodeError! Content is not JSON. Raw response content:")
        print(response.text) # This will show you if it's HTML or an error message
else:
    print(f"Server returned an error status: {response.status_code}")


name = ipni.search('Aspleniaceae')


def get_missing_orders(fams):
    ipni_ids = []
    for fam in fams:
        id = ipni.search(fam)
        result = next(id)

    orders = get_order_from_web(ipnis)
    return orders

def get_order_from_web(ipnis):
    orders = []
    for ipni in ipnis:
        url = f'https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:{ipni}'
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        div_element = soup.find('div',class_='cl-value-text')
        name = div_element.get_text()
        orders.append(name)

    return orders

orders = ['Aspleniaceae', 'Cephalotaxaceae', 'Cochlospermaceae', 'Cyatheaceae', 'Cystodiaceae', 'Dennstaedtiaceae', 'Dipteridaceae', 'Equisetaceae', 'Gleicheniaceae', 'Hymenophyllaceae', 'Hypodematiaceae', 'Impatiens', 'Isoetaceae', 'Lindsaeaceae', 'Lonchitidaceae', 'Lycopodiaceae', 'Marattiaceae', 'Marsileaceae', 'Matoniaceae', 'Ophioglossaceae', 'Osmundaceae', 'Podocarpaceae', 'Polypodiaceae', 'Psilotaceae', 'Pteridaceae', 'Saccolomataceae', 'Salviniaceae', 'Schizaeaceae', 'Sciadopityaceae', 'Selaginellaceae', 'Taxaceae', 'Thismiaceae', 'Welwitschiaceae', 'Zamiaceae']

out = get_missing_orders(orders)
print(out)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Trying the R download route to get taxonomic backbone

In [ ]:
import rpy2

